In [ ]:
import requests
import pandas as pd

# 1. Configuração (mantenha suas credenciais)
# inserir as informacoes de apikey e apitoekn

# 2. Função para chamar a API (evita repetir código)
def trello_get(endpoint, params=None):
    base_url = "https://api.trello.com/1"
    full_url = f"{base_url}/{endpoint}"
    auth = {'key': API_KEY, 'token': API_TOKEN}
    if params:
        auth.update(params)
    return requests.get(full_url, params=auth).json()

# 3. Buscar quadros e encontrar o desejado
boards = trello_get("members/me/boards")
board = next((b for b in boards if b['name'] == NOME_QUADRO), None)

print(f"✅ Conectado: {board['name']} (ID: {board['id']})")

# 4. Buscar cards e exportar
cards = trello_get(f"boards/{board['id']}/cards")
df_cards = pd.DataFrame(cards)
df_cards.to_csv("cards_trello.csv", index=False)

print(f"✅ {len(df_cards)} cards exportados para 'cards_trello.csv'")

In [ ]:
import requests
import pandas as pd
from datetime import datetime

# =====================================================
# CONFIGURAÇÃO
# =====================================================
# 1. Configuração (mantenha suas credenciais)
# inserir as informacoes de apikey e apitoekn

# =====================================================
# FUNÇÃO GENÉRICA PARA CHAMAR A API
# =====================================================
def trello_get(endpoint, params=None):
    base_url = "https://api.trello.com/1"
    full_url = f"{base_url}/{endpoint}"
    auth = {'key': API_KEY, 'token': API_TOKEN}
    if params:
        auth.update(params)
    response = requests.get(full_url, params=auth)
    response.raise_for_status()
    return response.json()

# =====================================================
# 1. BUSCAR QUADRO E CARDS
# =====================================================
boards = trello_get("members/me/boards")
board = next((b for b in boards if b['name'] == NOME_QUADRO), None)
if not board:
    raise Exception(f"Quadro '{NOME_QUADRO}' não encontrado")

print(f"✅ Conectado: {board['name']} (ID: {board['id']})")

cards = trello_get(f"boards/{board['id']}/cards")
print(f"✅ {len(cards)} cards encontrados")

# =====================================================
# 2. EXTRAIR AÇÕES (MOVIMENTAÇÕES) DOS CARDS
# =====================================================
acoes = []
for card in cards:
    card_actions = trello_get(f"cards/{card['id']}/actions", params={'filter': 'updateCard'})
    for act in card_actions:
        acoes.append({
            'card_id': card['id'],
            'card_name': card['name'],
            'action_date': act['date'],
            'list_before': act['data'].get('listBefore', {}).get('name'),
            'list_after': act['data'].get('listAfter', {}).get('name'),
            'action_type': act['type']
        })

df_acoes = pd.DataFrame(acoes)
print(f"✅ {len(df_acoes)} ações registradas")

# =====================================================
# 3. CALCULAR LEAD TIME (com card_id e metadados)
# =====================================================
lead_times = []
for card in cards:
    # Aproximação: usa 'dateLastActivity' como criação (não é 100% preciso)
    # Ideal: buscar a PRIMEIRA ação do card (tipo 'createCard')
    data_criacao = pd.to_datetime(card['dateLastActivity'])
    
    # Busca quando o card entrou na lista "Concluido"
    mov_concluido = df_acoes[
        (df_acoes['card_id'] == card['id']) & 
        (df_acoes['list_after'] == 'Concluido')
    ]
    
    if not mov_concluido.empty:
        data_conclusao = pd.to_datetime(mov_concluido.iloc[0]['action_date'])
        lead_time_dias = (data_conclusao - data_criacao).days
        status = 'Concluído'
    else:
        data_conclusao = None
        lead_time_dias = None
        status = 'Não concluído'
    
    lead_times.append({
        'card_id': card['id'],
        'card_name': card['name'],
        'data_criacao': data_criacao,
        'data_conclusao': data_conclusao,
        'lead_time_dias': lead_time_dias,
        'status': status
    })

df_lead = pd.DataFrame(lead_times)

# =====================================================
# 4. EXPORTAR ARQUIVOS
# =====================================================
# Cards (dados brutos)
df_cards = pd.DataFrame(cards)
df_cards.to_csv('cards_trello.csv', index=False)

# Ações (movimentações)
df_acoes.to_csv('acoes_trello.csv', index=False)

# Lead time (com card_id para relação)
df_lead.to_csv('lead_time.csv', index=False)

print("\n✅ Arquivos gerados:")
print("  - cards_trello.csv")
print("  - acoes_trello.csv")
print("  - lead_time.csv")

print("\n📊 Amostra do lead_time:")
print(df_lead[['card_name', 'lead_time_dias', 'status']].head(10))